In [ ]:
from argparse import ArgumentParser, Namespace
import Spectral_Preprocessing as sp
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
parser = ArgumentParser()
parser.add_argument("--range-min-ir", dest= "range_min_ir", type = int, default = 400)
parser.add_argument("--range-max-ir", dest= "range_max_ir", type = int, default = 4000)
parser.add_argument("--interpolate-interval-ir", dest = "interpolate_interval_ir", type = int, default = 1) 

parser.add_argument("--range-min-raman", dest= "range_min_raman", type = float, default = 400)
parser.add_argument("--range-max-raman", dest= "range_max_raman", type = float, default = 4000)
parser.add_argument("--interpolate-interval-raman", dest = "interpolate_interval_raman", type = float, default = 1) 

parser.add_argument("--rubberband", dest="rubberband", action="store_true", default=False,)
parser.add_argument("--minmax", dest="minmax", action="store_true", default=False,)
parser.add_argument("--no-interpolate", dest = "no_interpolate", action="store_true", default = False)
parser.add_argument("--nitrogen-calibration", dest="nitrogen_calibration", action="store_true", default=True,)
parser.add_argument("--si-peak", dest= "si_peak", type = float, default = 520)

parser.add_argument("--smooth", dest = "smooth", type = int, default = 0)
parser.add_argument("--baseline", dest = "baseline", type = int, default = 5)


In [ ]:
###Sample Code #1: Specify IR and Raman interpolation intervals along with min & max values
args: Namespace = parser.parse_args([]) ###Do not alter this section.
###PLEASE TYPE IN YOUR ARGUMENTS HERE
args.range_min_ir = 674.9918 ###enter an integer
args.range_max_ir = 3999.809 ###enter an integer [max value is not included]
args.interpolate_interval_ir = 1.929 ###enter an integer

args.range_min_raman = 355.555542 ###enter an integer or float
args.range_max_raman = 3895.935059 ##enter an integer or float [max value is not included]
args.interpolate_interval_raman = 2.168021680216802 ###enter an integer or float

args.rubberband = 'als' ###True for rubberband, False for iModPoly
args.minmax = False ###False for SVT normalization, True for minmax
args.no_interpolate = False ###use sample code 2 if using True for this option.
args.nitrogen_calibration = True ##True for calibration, False for no calibration.
args.si_peak = 520 ###enter an integer or float

#directory_name = '/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/MEC_02_115/Raw Data/' ###directory where the PTIR files will be downloaded from.
directory_name = 'MEC_02_115/Raw Data/' ###directory where the PTIR files will be downloaded from.
ir_export_filename =  "no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes
file_list_tmp = [] ###option to list out the files you want included. if left as-is, all files in directory will be included.

In [ ]:
path = 'MEC_02_115/Raw Data/'
dirs = os.listdir(path)

for d in dirs:
    if d[-4:] == 'ptir' or d==".DS_Store":
        del dirs[np.where(np.array(dirs) == d)[0][0]]

In [ ]:
files_list = []
for d in dirs:
    files = os.listdir(path + d)
    files_list = files_list + [d+'/'+f for f in files if f[-5:] == '.ptir']

In [ ]:
#files_list[1:]
files_list

In [ ]:
raman_final, ir_final, label_list, XX_ir, XX_raman, ids, coords, nitro_cal_list, rng = sp.ptir_to_interpolated_processed_csv(args, directory_name,
                                                            ir_export_filename, raman_export_filename, 
                                                            file_list = files_list[1:], transpose = True)


In [ ]:
ids_file_name = np.array([x[19:-5] 
                 if x[7] == '/' else x[20:-5] 
                 for x in ids])
ids_file_name = ids_file_name.astype('<U20')
ids_file_name_list = np.unique(ids_file_name)

idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    #ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1

ids_file_name

In [ ]:
preposs_data_raman = pd.DataFrame(raman_final)
preposs_data_raman.columns = XX_raman
preposs_data_raman = pd.concat([preposs_data_raman, coords], axis=1)
preposs_data_raman['id'] = ids_file_name
preposs_data_raman
#preposs_data_raman.to_csv()

In [ ]:
preposs_data_raman_cal = pd.DataFrame(raman_final)
preposs_data_raman_cal.columns = XX_raman
preposs_data_raman_cal = pd.concat([preposs_data_raman_cal, coords], axis=1)
preposs_data_raman_cal['id'] = ids_file_name
preposs_data_raman_cal
#preposs_data_raman.to_csv()

In [ ]:
ids_file_name

In [ ]:
ids_file_name_list.shape

In [ ]:
len(ids_file_name_list_)

In [ ]:
ids_file_name_list_ = []
for x in ids_file_name:
    if x not in ids_file_name_list_:
        ids_file_name_list_.append(x)
ids_file_name_list_

In [ ]:
nitro_cal_list

In [ ]:
nitro_cal = np.concatenate(nitro_cal_list, axis=0)
nitro_cal

In [ ]:
[(np.mean(x), np.std(x)) for x in nitro_cal_list]

In [ ]:
nitro_cal.mean(), nitro_cal.std()

In [ ]:
wn_2#[np.logical_and(wn_2 > 2280, wn_2 < 2380)]
#np.array(preposs_data_raman.query('id == @fn').iloc[:,:-3].loc[:,np.logical_and(wn_2 > 2280, wn_2 < 2380)]
 

In [ ]:
preposs_data_raman.shape

In [ ]:
x

In [ ]:
for i in range(1486):
    wn = np.float64(preposs_data_raman.columns[:-3]) 
    wn_diff = nitro_cal[i]
    idx = np.logical_and(wn > 2280, wn < 2380)
    x = wn[idx]
    y = list(preposs_data_raman.iloc[i,:-3].loc[idx])
    #if wn_diff is None:
    #    ya = y
    #elif wn_diff < 0:
    #    ya = y[abs(elements_to_chop):] + y[-1:]*abs(elements_to_chop)
        #wn_2 = abs(nitro_cal)*list(wn[:1])+list(wn[:nitro_cal])
    #elif wn_diff > 0:
    #    ya = y[:1]*elements_to_chop + y[:-elements_to_chop]
    #    wn_2 = list(wn[abs(nitro_cal):]) + abs(nitro_cal)*list(wn[:1])
    #else:
    #    wn_2 = wn.copy()
    #wn_2 = np.array(wn_2)
    _=plt.plot(x, y, alpha=.1, c='b')
    _=plt.plot(x-wn_diff, y, alpha=.1, c='r')
    _=plt.xlim(2280, 2380)
    if i>0 and preposs_data_raman['id'][i] != preposs_data_raman['id'][i+1]:
        _=plt.show()

In [ ]:
for i, fn in enumerate(ids_file_name_list_):
    wn = np.float64(preposs_data_raman.columns[:-3]) 
    wn_2 = np.float64(preposs_data_raman_cal.columns[:-3]) 
    _=plt.plot(wn[np.logical_and(wn > 2280, wn < 2380)], np.array(preposs_data_raman.query('id == @fn').iloc[:,:-3].loc[:,np.logical_and(wn > 2280, wn < 2380)]).T, alpha=.1, c='b')
    _=plt.plot(wn_2[np.logical_and(wn_2 > 2280, wn_2 < 2380)], np.array(preposs_data_raman_cal.query('id == @fn').iloc[:,:-3].loc[:,np.logical_and(wn_2 > 2280, wn_2 < 2380)]).T, alpha=.1, c='r')
#    _=plt.plot(wn_2[np.logical_and(wn_2 > 2280, wn_2 < 2380)], np.array(preposs_data_raman.query('id == @fn').iloc[:,:-3].loc[:,np.logical_and(wn_2 > 2280, wn_2 < 2380)]).T, alpha=.1, c='r')
    _=plt.show()

In [ ]:
for i, fn in enumerate(ids_file_name_list_):
    wn = np.float64(preposs_data_raman.columns[:-3]) 
    nitro_cal = nitro_cal_list[i]
    if nitro_cal is None:
        nitro_cal = 0
    if nitro_cal < 0:
        wn_2 = abs(nitro_cal)*list(wn[:1])+list(wn[:nitro_cal])
    elif nitro_cal > 0:
        wn_2 = list(wn[abs(nitro_cal):]) + abs(nitro_cal)*list(wn[:1])
    else:
        wn_2 = wn.copy()
    wn_2 = np.array(wn_2)
    _=plt.plot(wn[np.logical_and(wn > 2280, wn < 2380)], np.array(preposs_data_raman.query('id == @fn').iloc[:,:-3].loc[:,np.logical_and(wn > 2280, wn < 2380)]).T, alpha=.1, c='b')
    _=plt.plot(wn_2[np.logical_and(wn_2 > 2280, wn_2 < 2380)], np.array(preposs_data_raman.query('id == @fn').iloc[:,:-3].loc[:,np.logical_and(wn_2 > 2280, wn_2 < 2380)]).T, alpha=.1, c='r')
    _=plt.show()

In [ ]:
wn = np.float64(preposs_data_raman_cal.columns[:-3]) 
_=plt.plot(preposs_data_raman_cal.iloc[:,:-3].loc[:,np.logical_and(wn > 2280, wn < 2380)].T, alpha=.1, c='b')

In [ ]:
preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
preposs_data_ir['id'] = ids_file_name
preposs_data_ir

In [ ]:
plt.hist(np.abs(np.diff(X_ir,1)).mean(1))

In [ ]:
for i in np.argsort(np.abs(np.diff(X_ir,1)).mean(1)):
    plt.plot(X_ir[i,:])
    plt.plot(np.abs(np.diff(X_ir[i,:])))
    plt.show()
    plt.hist(np.abs(np.diff(X_ir,1))[i,:])
    plt.show()

In [ ]:
i = 210
plt.plot(X_ir[i,:])
plt.plot(np.abs(np.diff(X_ir[i,:])))
plt.show()
plt.hist(np.abs(np.diff(X_ir,1))[i,:])

In [ ]:
i = 210
(np.abs(np.diff(X_ir,1))[i,:].min(),
 np.quantile(np.abs(np.diff(X_ir,1))[i,:], .25),
 np.quantile(np.abs(np.diff(X_ir,1))[i,:], .5),
 np.abs(np.diff(X_ir,1))[i,:].mean(),
 np.abs(np.diff(X_ir,1))[i,:].std(),
 np.quantile(np.abs(np.diff(X_ir,1))[i,:], .75),
 np.abs(np.diff(X_ir,1))[i,:].max())

In [ ]:
i = 0
(np.abs(np.diff(X_ir,1))[i,:].min(),
 np.quantile(np.abs(np.diff(X_ir,1))[i,:], .25),
 np.quantile(np.abs(np.diff(X_ir,1))[i,:], .5),
 np.abs(np.diff(X_ir,1))[i,:].mean(),
 np.abs(np.diff(X_ir,1))[i,:].std(),
 np.quantile(np.abs(np.diff(X_ir,1))[i,:], .75),
 np.abs(np.diff(X_ir,1))[i,:].max())

In [ ]:
i = 0
plt.plot(X_ir[i,:])
plt.plot(np.abs(np.diff(X_ir[i,:])))
plt.show()
plt.hist(np.abs(np.diff(X_ir,1))[i,:])

In [ ]:
np.where(clust_ir == 1)

In [ ]:
plt.plot(preposs_data_ir.iloc[:,:-3].loc[clust_ir == 1,:].iloc[0,:])

In [ ]:
X_ir = np.array(preposs_data_ir.iloc[:,:-3])
#_ = plt.plot(preposs_data_ir.iloc[:,:-3].T)

wn = np.array(preposs_data_ir.columns[:-3])

idx = np.logical_or(np.logical_and(wn > 980, wn < 1800),
np.logical_and(wn > 2720, wn < 2970) #2993.64990234375-10
) 

X_ir_2 = X_ir[:,idx]
X_ir_2 = (X_ir_2 - X_ir_2.mean(1)[:,None]) / X_ir_2.std(1)[:,None]

from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=16, random_state=0).fit(X_ir)

clust_ir = kmeans.predict(X_ir)

In [ ]:
# [0, 2, 3, 6, 10, 12, 14, 15]

for i in range(16): #[0, 2, 3, 5, 7, 9, 12, 14, 15]
    if i in [0, 2, 3, 6, 10, 12, 14, 15]:
        continue
    _ = plt.plot(preposs_data_ir.iloc[:,:-3].loc[clust_ir == i,idx].T)
    _ = plt.show()

In [ ]:
for i in range(16): #[0, 2, 3, 5, 7, 9, 12, 14, 15]
    if i not in [0, 2, 3, 6, 10, 12, 14, 15]:
        continue
    _ = plt.plot(preposs_data_ir.iloc[:,:-3].loc[clust_ir == i,idx].T)
    _ = plt.show()

In [ ]:
for i in range(16): #[0, 2, 3, 5, 7, 9, 12, 14, 15]
    if i in [0, 2, 3, 6, 10, 12, 14, 15]:
        continue
    print(np.abs(np.diff(X_ir_2,1)).mean(1)[clust_ir == i].mean(),
          np.abs(np.diff(X_ir_2,1)).mean(1)[clust_ir == i].std())
    #_ = plt.plot(preposs_data_ir.iloc[:,:-3].loc[clust_ir == i,:].T)
    _ = plt.show()

In [ ]:
for i in range(16): #[0, 2, 3, 5, 7, 9, 12, 14, 15]
    if i not in [0, 2, 3, 6, 10, 12, 14, 15]:
        continue
    print(np.abs(np.diff(X_ir_2,1)).mean(1)[clust_ir == i].mean(),
          np.abs(np.diff(X_ir_2,1)).mean(1)[clust_ir == i].std())
    #_ = plt.plot(preposs_data_ir.iloc[:,:-3].loc[clust_ir == i,:].T)
    _ = plt.show()

In [ ]:
pd.DataFrame({'cluster':clust_ir,'n':1}).query('cluster in [0, 2, 3, 6, 10, 12, 14, 15]').sum()['n']

In [ ]:
_ = plt.plot(preposs_data_raman_cal.iloc[:,:-3].T)

In [ ]:
X_raman = np.array(preposs_data_raman_cal.iloc[:,:-3])
X_raman

In [ ]:
from scipy import stats

mode_ram = stats.mode(np.abs(np.diff(X_raman,1)), axis=1, keepdims=False)
mode_ram.count

In [ ]:
np.argsort(mode_ram.count)[::-1]

In [ ]:
for i in np.argsort(mode_ram.count)[::-1]:#np.argsort(np.abs(np.diff(X_raman,1)).mean(1)):
    plt.plot(X_raman[i,:])
    #plt.plot(np.abs(np.diff(X_ir[i,:])))
    plt.show()
    #plt.hist(np.abs(np.diff(X_ir,1))[i,:])
    #plt.show()